In [2]:
import json
import sys
import os
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib.cm as cm
from tqdm import tqdm
import pandas as pd
import autograd.numpy as np
import torch
from scipy.integrate import solve_ivp
import autograd
import math
import random

# PINNs

In [29]:
def load_train_data(data_dir):
    file_list = sorted([f for f in os.listdir(data_dir) if f.endswith(".json")])
    train_vo, train_ve, train_contact = [], [], []

    for fname in file_list:
        with open(os.path.join(data_dir, fname), 'r', encoding='utf-8') as f:
            data = json.load(f)

        frames = sorted(data.items(), key=lambda x: int(x[0]))
        vo_seq, ve_seq, contact_seq = [], [], []

        for _, frame in frames:
            vo_seq.append(frame["obj_6d"])
            ve_seq.append(frame["tcp_6d"])

            pts = frame["obj_tcp_pts_obj"]
            nmls = frame["obj_tcp_nmls_obj"]

            # 取前两个接触点（不足补零）
            contact_vec = []

            for i in range(2):
                if i < len(pts):
                    contact_vec += pts[i]
                    contact_vec += nmls[i]
                else:
                    contact_vec += [0, 0, 0]
                    contact_vec += [0, 0, 0]

            contact_seq.append(contact_vec)

        train_vo.append(torch.tensor(vo_seq))
        train_ve.append(torch.tensor(ve_seq))
        train_contact.append(torch.tensor(contact_seq))

    return train_vo, train_ve, train_contact

In [30]:
class w_Prediction(torch.nn.Module):
    def __init__(self, model, device='cuda'):
        super(w_Prediction, self).__init__()
        self.model = model.to(device)
        self.device = device

    def forward(self, contact_infos, obj_velocity, ee_velocity):
        x = torch.cat(
            [contact_infos, obj_velocity, ee_velocity],
            dim=0
        )                
        x = x.unsqueeze(0).to(self.device).float()
        v_rot= self.model(x)
        wz_pred = v_rot.squeeze() 
        return wz_pred  

    
    def train_model(self, train_vo, train_ve, train_contact, val_vo, val_ve, val_contact,
                    lr=5e-5, epochs=200, patience=10, verbose=False, print_step=10, weight_decay=1e-4):

        optim = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)
        stats = {"loss": [], "val_loss": []}
        best_val_loss = float("inf")
        counter = 0

        for epoch in range(epochs):
            self.model.train()
            epoch_loss = 0.0

            for i in range(len(train_vo)):
                sample_vo = train_vo[i].to(self.device).float()
                sample_ve = train_ve[i].to(self.device).float()
                sample_contact = train_contact[i].to(self.device).float()
                n_frames = sample_vo.shape[0]
                loss = 0.0
                for t in range(1, n_frames - 1):                   
                    vo_t = sample_vo[t]
                    ve_t = sample_ve[t]
                    contact_t = sample_contact[t]
                    vo_true = sample_vo[t + 1]
                    wz_pre = self.forward(contact_t, vo_t, ve_t)
                    trans_true = vo_true[:3]
                    rot_true  = vo_true[3:]
                    wz_true = rot_true[0]
                    rot_pre = torch.zeros_like(trans_true)
                    rot_pre[0] = wz_pre
                    ve_true = sample_ve[t+1][:3]
                    pe = sample_contact[t+1][:3]         # contact point (A frame)
                    ne = sample_contact[t+1][3:6]         # normal (A frame, unit)
                    vo_e =  trans_true + torch.cross(rot_pre, pe)
                    vn_e = ((vo_e - ve_true) * ne).sum(dim=-1)
                    r = 0.1
                    loss += (wz_pre - wz_true).pow(2) + r * 1e2 * (vn_e).pow(2)
                loss /= (n_frames - 2)
                optim.zero_grad()
                loss.backward()
                optim.step()
                epoch_loss += loss.item()

            avg_loss = epoch_loss / len(train_vo)
            stats["loss"].append(avg_loss)

            self.model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for i in range(len(val_vo)):
                    sample_vo = val_vo[i].to(self.device).float()
                    sample_ve = val_ve[i].to(self.device).float()
                    sample_contact = val_contact[i].to(self.device).float()

                    n_frames = sample_vo.shape[0]
                    loss = 0.0
                    for t in range(1, n_frames - 1):
                        vo_t = sample_vo[t]
                        ve_t = sample_ve[t]
                        contact_t = sample_contact[t]
                        vo_true = sample_vo[t + 1]
                        wz_pre = self.forward(contact_t, vo_t, ve_t)
                        trans_true = vo_true[:3]
                        rot_true  = vo_true[3:]
                        wz_true = rot_true[0]
                        rot_pre = torch.zeros_like(trans_true)
                        rot_pre[0] = wz_pre
                        ve_true = sample_ve[t+1][:3]
                        pe = sample_contact[t+1][:3]         # contact point (A frame)
                        ne = sample_contact[t+1][3:6]         # normal (A frame, unit)
                        vo_e =  trans_true + torch.cross(rot_pre, pe)
                        vn_e = ((vo_e - ve_true) * ne).sum(dim=-1)
                        r = 0.1
                        loss += (wz_pre - wz_true).pow(2) + r * 1e2 * (vn_e).pow(2)
                    loss /= (n_frames - 2)
                    val_loss += (loss).item()

            val_loss /= len(val_vo)
            stats["val_loss"].append(val_loss)

            if verbose and epoch % print_step == 0:
                print(f"Epoch {epoch}, Train Loss: {avg_loss:.6f}, Val Loss: {val_loss:.6f}")

            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                counter = 0
            else:
                counter += 1
                if counter >= patience:
                    if verbose:
                        print(f"Early stopping at epoch {epoch}")
                    break

        return stats

    def test_model(self, test_vo, test_ve, test_contact):
        self.model.eval()
        loss = 0.0

        with torch.no_grad():
            for i in range(len(test_vo)):
                sample_vo = test_vo[i].to(self.device).float()
                sample_ve = test_ve[i].to(self.device).float()
                sample_contact = test_contact[i].to(self.device).float()

                n_frames = sample_vo.shape[0]
                loss_pre = 0.0
                for t in range(1, n_frames - 1):   
                    vo_t = sample_vo[t]
                    ve_t = sample_ve[t]
                    contact_t = sample_contact[t]
                    vo_true = sample_vo[t + 1]
                    wz = self.forward(contact_t, vo_t, ve_t)
                    rot_true   = vo_true[3:]
                    loss_pre += (wz - rot_true[0]).pow(2)
                loss_pre /= (n_frames - 2)
                loss += loss_pre
        return loss / len(test_vo)

In [31]:
class v_Prediction(torch.nn.Module):
    def __init__(self, model, device='cuda'):
        super(v_Prediction, self).__init__()
        self.model = model.to(device)
        self.device = device

    def forward(self, contact_infos, obj_velocity, ee_velocity):
        x = torch.cat(
            [contact_infos, obj_velocity, ee_velocity],
            dim=0
        )                
        x = x.unsqueeze(0).to(self.device).float()
        v_trans = self.model(x)
        return v_trans.squeeze(0) 
    
    def train_model(self, train_vo, train_ve, train_contact, val_vo, val_ve, val_contact,
                    lr=5e-5, epochs=200, patience=10, verbose=False, print_step=10, weight_decay=1e-4):

        optim = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)
        stats = {"loss": [], "val_loss": []}
        best_val_loss = float("inf")
        counter = 0

        for epoch in range(epochs):
            self.model.train()
            epoch_loss = 0.0

            for i in range(len(train_vo)):
                sample_vo = train_vo[i].to(self.device).float()
                sample_ve = train_ve[i].to(self.device).float()
                sample_contact = train_contact[i].to(self.device).float()
                n_frames = sample_vo.shape[0]
                loss = 0.0
                for t in range(1, n_frames - 1):
                    vo_t = sample_vo[t]
                    ve_t = sample_ve[t]
                    contact_t = sample_contact[t]
                    vo_true = sample_vo[t + 1]
                    vxy = self.forward(contact_t, vo_t, ve_t)
                    trans_true = vo_true[:3]
                    rot_true  = vo_true[3:]
                    vo_pre = torch.zeros_like(trans_true)
                    vo_pre[1] = vxy[0]
                    vo_pre[2] = vxy[1]
                    ve_true = sample_ve[t+1][:3]
                    pe = sample_contact[t+1][:3]         # contact point (A frame)
                    ne = sample_contact[t+1][3:6]         # normal (A frame, unit)
                    vo_e =  vo_pre + torch.cross(rot_true, pe)
                    vn_e = ((vo_e - ve_true) * ne).sum(dim=-1)
                    r = 0.1
                    loss += 1e4 * (vo_pre - trans_true).pow(2).mean() + r * 1e4 * (vn_e).pow(2).mean()
                loss /= (n_frames - 2)
                optim.zero_grad()
                loss.backward()
                optim.step()
                epoch_loss += loss.item()

            avg_loss = epoch_loss / len(train_vo)
            stats["loss"].append(avg_loss)

            self.model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for i in range(len(val_vo)):
                    sample_vo = val_vo[i].to(self.device).float()
                    sample_ve = val_ve[i].to(self.device).float()
                    sample_contact = val_contact[i].to(self.device).float()

                    n_frames = sample_vo.shape[0]
                    loss = 0.0
                    for t in range(1, n_frames - 1):
                        vo_t = sample_vo[t]
                        ve_t = sample_ve[t]
                        contact_t = sample_contact[t]
                        vo_true = sample_vo[t + 1]
                        vxy = self.forward(contact_t, vo_t, ve_t)
                        trans_true = vo_true[:3]
                        rot_true  = vo_true[3:]
                        vo_pre = torch.zeros_like(trans_true)
                        vo_pre[1] = vxy[0]
                        vo_pre[2] = vxy[1]
                        ve_true = sample_ve[t+1][:3]
                        pe = sample_contact[t+1][:3]         # contact point (A frame)
                        ne = sample_contact[t+1][3:6]         # normal (A frame, unit)
                        vo_e =  vo_pre + torch.cross(rot_true, pe)
                        vn_e = ((vo_e - ve_true) * ne).sum(dim=-1)
                        r = 0.1
                        loss += 1e4 * (vo_pre - trans_true).pow(2).mean() + r * 1e4 * (vn_e).pow(2).mean()
                    loss /= (n_frames - 2)
                    val_loss += (loss).item()

            val_loss /= len(val_vo)
            stats["val_loss"].append(val_loss)

            if verbose and epoch % print_step == 0:
                print(f"Epoch {epoch}, Train Loss: {avg_loss:.6f}, Val Loss: {val_loss:.6f}")

            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                counter = 0
            else:
                counter += 1
                if counter >= patience:
                    if verbose:
                        print(f"Early stopping at epoch {epoch}")
                    break

        return stats

    def test_model(self, test_vo, test_ve, test_contact):
        self.model.eval()
        loss = 0.0

        with torch.no_grad():
            for i in range(len(test_vo)):
                sample_vo = test_vo[i].to(self.device).float()
                sample_ve = test_ve[i].to(self.device).float()
                sample_contact = test_contact[i].to(self.device).float()

                n_frames = sample_vo.shape[0]
                loss_pre = 0.0
                for t in range(1, n_frames - 1):   
                    vo_t = sample_vo[t]
                    ve_t = sample_ve[t]
                    contact_t = sample_contact[t]
                    vo_true = sample_vo[t + 1]
                    vo_pre = self.forward(contact_t, vo_t, ve_t)
                    trans_true = vo_true[1:3]
                    loss_pre += 1e4 * (vo_pre - trans_true).pow(2).mean()
                loss_pre /= (n_frames - 2)
                loss += loss_pre
        return loss / len(test_vo)

In [32]:
class MLP_w(torch.nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 128, 64)):
        super().__init__()

        layers = []
        last_dim = input_dim
        for h in hidden_dims:
            layers.append(torch.nn.Linear(last_dim, h))
            layers.append(torch.nn.Tanh())
            last_dim = h

        self.shared = torch.nn.Sequential(*layers)
        self.fc_rot   = torch.nn.Linear(last_dim, 1)  # 旋转

    def forward(self, x):
        h = self.shared(x)
        v_rot   = self.fc_rot(h)
        return v_rot

In [33]:
class MLP_v(torch.nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 128, 64)):
        super().__init__()

        layers = []
        last_dim = input_dim
        for h in hidden_dims:
            layers.append(torch.nn.Linear(last_dim, h))
            layers.append(torch.nn.Tanh())
            last_dim = h

        self.shared = torch.nn.Sequential(*layers)
        self.fc_trans = torch.nn.Linear(last_dim, 2)  # 平移

    def forward(self, x):
        h = self.shared(x)
        v_trans = self.fc_trans(h)
        return v_trans

In [11]:
import os
import json
import torch
import numpy as np
import random

def set_seed(seed=2023):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def split_train_val(train_vo, train_ve, train_contact):
    n = len(train_vo) - 15 # test
    n_train = n - 5  # 最后几个做验证

    train_data = (
        train_vo[:n_train],
        train_ve[:n_train],
        train_contact[:n_train]
    )

    test_data = (
        train_vo[n_train:n],
        train_ve[n_train:n],
        train_contact[n_train:n]
    )
    
    val_data = (
        train_vo[n:],
        train_ve[n:],
        train_contact[n:]
    )
    return train_data, val_data, test_data

set_seed(2023)

data_dir = r"C:\Users\30567\Desktop\wk\PINNs-VLA\PINNs_WM\irregular_T_1\train_data"

train_vo, train_ve, train_contact = load_train_data(data_dir)
(train_vo, train_ve, train_contact), (val_vo, val_ve, val_contact), (test_vo, test_ve, test_contact)= \
    split_train_val(train_vo, train_ve, train_contact)

input_dim = 12 + 6 + 6  # 24

net = MLP_v(
    input_dim=input_dim,
    hidden_dims=(32, 32, 32, 32, 32, 32, 32, 32, 32, 32)
)

pinn = v_Prediction(net, device='cpu')


model_dir = "models"
os.makedirs(model_dir, exist_ok=True)
filepath = os.path.join(model_dir, "v_prediction.pth")


print("Start training (will overwrite existing model if any)...")

stats = pinn.train_model(
    train_vo, train_ve, train_contact,
    val_vo, val_ve, val_contact,
    lr=1e-4,
    epochs=300,
    patience=20,
    verbose=True,
    print_step=10
)

torch.save(pinn.state_dict(), filepath)
print(f"Model saved to {filepath}") 
test_loss = pinn.test_model(
    test_vo=test_vo, 
    test_ve=test_ve,
    test_contact = test_contact
)
print(f"Test Loss: {test_loss:.6f}")


Start training (will overwrite existing model if any)...


C:\Users\30567\AppData\Local\Temp\ipykernel_23188\548000470.py:48: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Cross.cpp:67.)
  vo_e =  vo_pre + torch.cross(rot_true, pe)


Epoch 0, Train Loss: 72.842907, Val Loss: 29.390692
Epoch 10, Train Loss: 6.076286, Val Loss: 6.027172
Epoch 20, Train Loss: 3.615973, Val Loss: 3.888451
Epoch 30, Train Loss: 3.275754, Val Loss: 3.582082
Epoch 40, Train Loss: 2.675589, Val Loss: 2.821145
Epoch 50, Train Loss: 1.837173, Val Loss: 1.733149
Epoch 60, Train Loss: 1.735062, Val Loss: 1.636885
Epoch 70, Train Loss: 1.645178, Val Loss: 1.536107
Epoch 80, Train Loss: 1.547477, Val Loss: 1.419007
Epoch 90, Train Loss: 1.422187, Val Loss: 1.265035
Epoch 100, Train Loss: 1.231695, Val Loss: 1.033715
Epoch 110, Train Loss: 0.913349, Val Loss: 0.686836
Epoch 120, Train Loss: 0.696871, Val Loss: 0.538767
Epoch 130, Train Loss: 0.657670, Val Loss: 0.498471
Epoch 140, Train Loss: 0.633718, Val Loss: 0.467665
Epoch 150, Train Loss: 0.614851, Val Loss: 0.445219
Epoch 160, Train Loss: 0.598762, Val Loss: 0.428229
Epoch 170, Train Loss: 0.584363, Val Loss: 0.415154
Epoch 180, Train Loss: 0.571061, Val Loss: 0.404586
Epoch 190, Train Loss

In [42]:
set_seed(2023)

data_dir = r"C:\Users\30567\Desktop\wk\PINNs-VLA\PINNs_WM\irregular_T_1\train_data"

train_vo, train_ve, train_contact = load_train_data(data_dir)
(train_vo, train_ve, train_contact), (val_vo, val_ve, val_contact), (test_vo, test_ve, test_contact) = \
    split_train_val(train_vo, train_ve, train_contact)

input_dim = 12 + 6 + 6  # 24

net = MLP_v(
    input_dim=input_dim,
    hidden_dims=(32, 32, 32, 32, 32, 32, 32, 32, 32, 32)
)

pinn = v_Prediction(net, device='cpu')

model_dir = "models"
os.makedirs(model_dir, exist_ok=True)
filepath = os.path.join(model_dir, "v_prediction.pth")

if os.path.exists(filepath):
    print(f"Found existing model, loading from {filepath}")
    
    # 1. 读取旧 state
    old_state = torch.load(filepath, map_location='cpu')
    
    # 2. 新 dict
    new_state = {}
    for k, v in old_state.items():
        # 如果 key 里有 model. 开头，要去掉
        if k.startswith("model."):
            k = k[len("model."):]

        new_state[k] = v

    # 3. 只 load 到 model
    pinn.model.load_state_dict(new_state)
    print("Model loaded, continue training")
else:
    print("No existing model found, training from scratch")


print("Start training...")

stats = pinn.train_model(
    train_vo, train_ve, train_contact,
    val_vo, val_ve, val_contact,
    lr=1e-5,
    epochs=100,
    patience=20,
    verbose=True,
    print_step=10
)

torch.save(pinn.state_dict(), filepath)
print(f"Model saved to {filepath}")

test_loss = pinn.test_model(
    test_vo=test_vo,
    test_ve=test_ve,
    test_contact=test_contact
)
print(f"Test Loss: {test_loss:.6f}")


Found existing model, loading from models\v_prediction.pth
Model loaded, continue training
Start training...
Epoch 0, Train Loss: 0.422156, Val Loss: 0.250480
Epoch 10, Train Loss: 0.422300, Val Loss: 0.249598
Epoch 20, Train Loss: 0.422139, Val Loss: 0.249531
Early stopping at epoch 21
Model saved to models\v_prediction.pth
Test Loss: 0.104430


In [15]:
import os
import json
import torch
import numpy as np
import random

def set_seed(seed=2023):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def split_train_val(train_vo, train_ve, train_contact):
    n = len(train_vo) - 15 # test
    n_train = n - 5  # 最后几个做验证

    train_data = (
        train_vo[:n_train],
        train_ve[:n_train],
        train_contact[:n_train]
    )

    test_data = (
        train_vo[n_train:n],
        train_ve[n_train:n],
        train_contact[n_train:n]
    )
    
    val_data = (
        train_vo[n:],
        train_ve[n:],
        train_contact[n:]
    )
    return train_data, val_data, test_data                                                                                                                                                                                                                                                                          

set_seed(2023)

data_dir = r"C:\Users\30567\Desktop\wk\PINNs-VLA\PINNs_WM\irregular_T_1\train_data"

train_vo, train_ve, train_contact = load_train_data(data_dir)
(train_vo, train_ve, train_contact), (val_vo, val_ve, val_contact), (test_vo, test_ve, test_contact)= \
    split_train_val(train_vo, train_ve, train_contact)

input_dim = 12 + 6 + 6  # 24

net = MLP_w(
    input_dim=input_dim,
    hidden_dims=(32, 32, 32, 32, 32, 32, 32, 32, 32, 32)
)

pinn = w_Prediction(net, device='cpu')


model_dir = "models"
os.makedirs(model_dir, exist_ok=True)
filepath = os.path.join(model_dir, "w_prediction.pth")


print("Start training (will overwrite existing model if any)...")

stats = pinn.train_model(
    train_vo, train_ve, train_contact,
    val_vo, val_ve, val_contact,
    lr=1e-4,
    epochs=300,
    patience=20,
    verbose=True,
    print_step=10
)

torch.save(pinn.state_dict(), filepath)
print(f"Model saved to {filepath}") 
test_loss = pinn.test_model(
    test_vo=test_vo, 
    test_ve=test_ve,
    test_contact = test_contact
)
print(f"Test Loss: {test_loss:.6f}")


Start training (will overwrite existing model if any)...
Epoch 0, Train Loss: 0.485648, Val Loss: 0.421515
Epoch 10, Train Loss: 0.067154, Val Loss: 0.054493
Epoch 20, Train Loss: 0.032971, Val Loss: 0.029327
Epoch 30, Train Loss: 0.025233, Val Loss: 0.023134
Epoch 40, Train Loss: 0.022179, Val Loss: 0.020678
Epoch 50, Train Loss: 0.020569, Val Loss: 0.019452
Epoch 60, Train Loss: 0.019559, Val Loss: 0.018721
Epoch 70, Train Loss: 0.018859, Val Loss: 0.018239
Epoch 80, Train Loss: 0.018340, Val Loss: 0.017898
Epoch 90, Train Loss: 0.017934, Val Loss: 0.017646
Epoch 100, Train Loss: 0.017601, Val Loss: 0.017449
Epoch 110, Train Loss: 0.017317, Val Loss: 0.017288
Epoch 120, Train Loss: 0.017064, Val Loss: 0.017148
Epoch 130, Train Loss: 0.016828, Val Loss: 0.017015
Epoch 140, Train Loss: 0.016595, Val Loss: 0.016873
Epoch 150, Train Loss: 0.016353, Val Loss: 0.016698
Epoch 160, Train Loss: 0.016098, Val Loss: 0.016467
Epoch 170, Train Loss: 0.015852, Val Loss: 0.016200
Epoch 180, Train L

In [45]:
set_seed(2023)

data_dir = r"C:\Users\30567\Desktop\wk\PINNs-VLA\PINNs_WM\irregular_T_1\train_data"

train_vo, train_ve, train_contact = load_train_data(data_dir)
(train_vo, train_ve, train_contact), (val_vo, val_ve, val_contact), (test_vo, test_ve, test_contact) = \
    split_train_val(train_vo, train_ve, train_contact)

input_dim = 12 + 6 + 6  # 24

net = MLP_w(
    input_dim=input_dim,
    hidden_dims=(32, 32, 32, 32, 32, 32, 32, 32, 32, 32)
)

pinn = w_Prediction(net, device='cpu')

model_dir = "models"
os.makedirs(model_dir, exist_ok=True)
filepath = os.path.join(model_dir, "w_prediction.pth")

if os.path.exists(filepath):
    print(f"Found existing model, loading from {filepath}")
    
    # 1. 读取旧 state
    old_state = torch.load(filepath, map_location='cpu')
    
    # 2. 新 dict
    new_state = {}
    for k, v in old_state.items():
        # 如果 key 里有 model. 开头，要去掉
        if k.startswith("model."):
            k = k[len("model."):]

        new_state[k] = v

    # 3. 只 load 到 model
    pinn.model.load_state_dict(new_state)
    print("Model loaded, continue training")
else:
    print("No existing model found, training from scratch")


print("Start training...")

stats = pinn.train_model(
    train_vo, train_ve, train_contact,
    val_vo, val_ve, val_contact,
    lr=1e-5,
    epochs=100,
    patience=20,
    verbose=True,
    print_step=10
)

torch.save(pinn.state_dict(), filepath)
print(f"Model saved to {filepath}")

test_loss = pinn.test_model(
    test_vo=test_vo,
    test_ve=test_ve,
    test_contact=test_contact
)
print(f"Test Loss: {test_loss:.6f}")


Found existing model, loading from models\w_prediction.pth
Model loaded, continue training
Start training...
Epoch 0, Train Loss: 0.014284, Val Loss: 0.013850
Epoch 10, Train Loss: 0.014271, Val Loss: 0.013835
Epoch 20, Train Loss: 0.014268, Val Loss: 0.013833
Epoch 30, Train Loss: 0.014265, Val Loss: 0.013830
Epoch 40, Train Loss: 0.014262, Val Loss: 0.013828
Epoch 50, Train Loss: 0.014260, Val Loss: 0.013826
Epoch 60, Train Loss: 0.014257, Val Loss: 0.013824
Epoch 70, Train Loss: 0.014254, Val Loss: 0.013822
Epoch 80, Train Loss: 0.014251, Val Loss: 0.013820
Epoch 90, Train Loss: 0.014249, Val Loss: 0.013818
Model saved to models\w_prediction.pth
Test Loss: 0.004219


# Pos Prediction

In [ ]:
import os, json, numpy as np
import pybullet as p
from scipy.spatial.transform import Rotation as R
from scipy.interpolate import CubicSpline
import torch

# =========================
# 1. 网络定义（不动）
# =========================
class MLP_v(torch.nn.Module):
    def __init__(self, input_dim=24, hidden_dims=(32, 32, 32, 32, 32, 32, 32, 32, 32, 32)):
        super().__init__()
        layers = []
        last = input_dim
        for h in hidden_dims:
            layers += [torch.nn.Linear(last, h), torch.nn.Tanh()]
            last = h
        self.shared = torch.nn.Sequential(*layers)
        self.fc_v = torch.nn.Linear(last, 2)

    def forward(self, x):
        h = self.shared(x)
        return self.fc_v(h)
    
class MLP_w(torch.nn.Module):
    def __init__(self, input_dim=24, hidden_dims=(32, 32, 32, 32, 32, 32, 32, 32, 32, 32)):
        super().__init__()
        layers = []
        last = input_dim
        for h in hidden_dims:
            layers += [torch.nn.Linear(last, h), torch.nn.Tanh()]
            last = h
        self.shared = torch.nn.Sequential(*layers)
        self.fc_w = torch.nn.Linear(last, 1)

    def forward(self, x):
        h = self.shared(x)
        return self.fc_w(h)

class v_Prediction(torch.nn.Module):
    def __init__(self, net, device="cpu"):
        super().__init__()
        self.net = net.to(device)
        self.device = device

    def forward(self, contact, vo, ve):
        x = torch.cat([contact, vo, ve], dim=0).unsqueeze(0).to(self.device).float()
        v = self.net(x)
        return v.squeeze(0)

class w_Prediction(torch.nn.Module):
    def __init__(self, net, device="cpu"):
        super().__init__()
        self.net = net.to(device)
        self.device = device

    def forward(self, contact, vo, ve):
        x = torch.cat([contact, vo, ve], dim=0).unsqueeze(0).to(self.device).float()
        w = self.net(x)
        return w.squeeze(0)

# =========================
# 2. 模型加载（不动）
# =========================
@torch.no_grad()
def load_v_prediction_model(path, device="cpu"):
    net = MLP_v()
    model = v_Prediction(net, device)
    state_dict = torch.load(path, map_location=device)
    new_state = {}
    for k, v in state_dict.items():
        if k.startswith("model."): k = k[len("model."):]
        if k.startswith("net."): k = k[len("net."):]
        k = k.replace("fc_trans", "fc_v")
        new_state["net."+k] = v
    model.load_state_dict(new_state, strict=True)
    model.eval()
    return model

@torch.no_grad()
def load_w_prediction_model(path, device="cpu"):
    net = MLP_w()
    model = w_Prediction(net, device)
    state_dict = torch.load(path, map_location=device)
    new_state = {}
    for k, v in state_dict.items():
        if k.startswith("model."): k = k[len("model."):]
        if k.startswith("net."): k = k[len("net."):]
        k = k.replace("fc_rot", "fc_w")
        new_state["net."+k] = v
    model.load_state_dict(new_state, strict=True)
    model.eval()
    return model

# =========================
# 3. 推理函数（不动）
# =========================
@torch.no_grad()
def infer_pose(xt, at, contact_info, model_v, model_w, dt, device='cpu'):
    p_w = xt[0:3]
    r_w = xt[3:6]
    v_w = xt[6:9]
    w_w = xt[9:12]

    v_tcp_w = at[6:9]
    w_tcp_w = at[9:12]

    R_wb = R.from_rotvec(r_w)

    # OBJ 和 TCP 在 body frame 下的速度
    vo = np.hstack([R_wb.inv().apply(v_w), R_wb.inv().apply(w_w)])
    ve = np.hstack([R_wb.inv().apply(v_tcp_w), R_wb.inv().apply(w_tcp_w)])

    # Contact 转到 body frame
    contact_b = []
    for i in range(2):
        pw = contact_info[i*6:i*6+3]
        nw = contact_info[i*6+3:i*6+6]
        contact_b += list(R_wb.inv().apply(pw - p_w))
        contact_b += list(R_wb.inv().apply(nw))
    contact_b = torch.tensor(contact_b, dtype=torch.float32, device=device)
    vo = torch.tensor(vo, dtype=torch.float32, device=device)
    ve = torch.tensor(ve, dtype=torch.float32, device=device)

    # 预测 v 和 w
    vxy_pred = model_v(contact_b, vo, ve).cpu().numpy()
    wz_pred = model_w(contact_b, vo, ve).item()  # 标量
    w_next_b = np.zeros(3, dtype=np.float32)
    v_next_b = np.zeros(3, dtype=np.float32)
    w_next_b[0] = wz_pred
    v_next_b[1] = vxy_pred[0]
    v_next_b[2] = vxy_pred[1]

    # 转回世界坐标
    v_next_w = R_wb.apply(v_next_b)
    w_next_w = R_wb.apply(w_next_b)

    # 构造下一帧状态
    xt_next = np.zeros(12, dtype=np.float32)
    xt_next[0:3] = p_w + v_next_w * dt
    xt_next[3:6] = (R.from_rotvec(w_next_w * dt) * R_wb).as_rotvec()
    xt_next[6:9] = v_next_w
    xt_next[9:12] = w_next_w

    return xt_next

# =========================
# 4. 读取 JSON
# =========================
json_path = r"C:\Users\30567\Desktop\wk\PINNs-VLA\PINNs_WM\regular_T\input_data\trial_11.json"
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

frames = sorted(data.values(), key=lambda x: x["time"])
times = np.array([f["time"] for f in frames])
dt = times[1] - times[0]

tcp_pos = np.array([f["tcp"]["sixd"][:3] for f in frames])
tcp_rotvec = np.array([f["tcp"]["sixd"][3:] for f in frames])
tcp_quat = R.from_rotvec(tcp_rotvec).as_quat()

obj_pos_gt = np.array([f["object"]["sixd"][:3] for f in frames])
obj_rotvec_gt = np.array([f["object"]["sixd"][3:] for f in frames])

# =========================
# 5. TCP 速度 / 角速度（不动）
# =========================
tcp_pos_spline = CubicSpline(times, tcp_pos, axis=0)
tcp_v = tcp_pos_spline(times, 1)

tcp_w = np.zeros_like(tcp_pos)
for k in range(len(times)-1):
    dq = R.from_quat(tcp_quat[k+1]) * R.from_quat(tcp_quat[k]).inv()
    tcp_w[k] = dq.as_rotvec() / dt
tcp_w[-1] = tcp_w[-2]

# =========================
# 6. OBJ xt（差分得到速度）
# =========================
xt_gt = np.zeros((len(times), 12))
xt_gt[:, 0:3] = obj_pos_gt
xt_gt[:, 3:6] = obj_rotvec_gt

for k in range(1, len(times)):
    xt_gt[k, 6:9] = (obj_pos_gt[k] - obj_pos_gt[k-1]) / dt
    dR = R.from_rotvec(obj_rotvec_gt[k]) * R.from_rotvec(obj_rotvec_gt[k-1]).inv()
    xt_gt[k, 9:12] = dR.as_rotvec() / dt

xt_gt[0, 6:12] = 0.0

# =========================
# 7. PyBullet 初始化
# =========================
if p.isConnected():
    p.disconnect()
p.connect(p.DIRECT)
p.setGravity(0, 0, 0)

obj_path = r"C:\Users\30567\Desktop\wk\PINNs-VLA\PINNs_WM\regular_T\object\object.obj"
obj_id = p.createMultiBody(
    baseMass=1.0,
    baseCollisionShapeIndex=p.createCollisionShape(p.GEOM_MESH, fileName=obj_path),
    baseVisualShapeIndex=p.createVisualShape(p.GEOM_MESH, fileName=obj_path),
)

tcp_id = p.createMultiBody(
    baseMass=0,
    baseCollisionShapeIndex=p.createCollisionShape(p.GEOM_CYLINDER, radius=0.005, height=0.05),
    baseVisualShapeIndex=p.createVisualShape(p.GEOM_CYLINDER, radius=0.005, length=0.05),
)

ground_id = p.createMultiBody(
    baseMass=0,
    baseCollisionShapeIndex=p.createCollisionShape(p.GEOM_BOX, halfExtents=[2, 2, 0.005]),
    basePosition=[0, 0, -0.015]
)

# =========================
# 8. 预测速度修正（防止穿模）
# =========================
@torch.no_grad()
def correct_pred(xt_pred, contact_info):
    obj_pos = xt_pred[0:3]
    obj_rot = xt_pred[3:6]
    obj_v = xt_pred[6:9]
    obj_w = xt_pred[9:12]
    # 修正
    if contact_info[6] > 0.001:
        p_c = contact_info[0:3]  # 接触点（世界坐标）
        p_o = obj_pos  # 物体中心
        r = p_c - p_o
        v_contact = obj_v + np.cross(obj_w, r)  # 接触点速度（刚体公式）
        n = contact_info[3:6]  # 法向量
        v_n_scalar = np.dot(v_contact, n) # 法向速度（标量：投影大小）
        if v_n_scalar <= 1e-1 * np.sqrt(np.dot(v_contact, v_contact)):
            xt_pred[0:3] = xt_pred[0:3] + n * contact_info[6]
            return xt_pred
        #  vo_new = (1 + a) * vo
        #  (vo_new * dt - vo * dt) * n = c = a * (vo * dt) * n 
        #  (a * vo) * n = a * (vo * n) = a vn
        #  a * vn * dt = c
        #  a = c / (vn * dt)
        # 构造下一帧状态
        xt_pred[0:3] = obj_pos + obj_v * contact_info[6]/v_n_scalar
        xt_pred[3:6] = (R.from_rotvec(obj_w * (contact_info[6]/(v_n_scalar))) * R.from_rotvec(obj_rot)).as_rotvec()
        xt_pred[6:9] = obj_v * (1 + (contact_info[6]/(v_n_scalar/24.0)))
        xt_pred[9:12] = obj_w * (1 + (contact_info[6]/(v_n_scalar/24.0)))
    return xt_pred

# =========================
# 9. Rollout（OBJ 用 GT，TCP 用样条）改版
# =========================
model_v = load_v_prediction_model("models/v_prediction.pth", device='cpu')
model_w = load_w_prediction_model("models/w_prediction.pth", device='cpu')

for k in range(len(times) - 1):
    xt = xt_gt[k].copy()

    at = np.zeros(12)
    at[6:9] = tcp_v[k]
    at[9:12] = tcp_w[k]

    # 更新 PyBullet
    p.resetBasePositionAndOrientation(
        obj_id, obj_pos_gt[k], R.from_rotvec(obj_rotvec_gt[k]).as_quat()
    )
    p.resetBasePositionAndOrientation(
        tcp_id, tcp_pos[k], tcp_quat[k]
    )
    p.performCollisionDetection()

    # 获取接触信息
    contact_info = np.zeros(14, dtype=np.float32)
    c1 = p.getClosestPoints(obj_id, tcp_id, distance=0.002)
    if len(c1) > 0:
        contact_info[0:3] = c1[0][5]  # 接触点
        contact_info[3:6] = c1[0][7]  # 法向量
        contact_info[6]   = -c1[0][8]  # 接触深度
    c2 = p.getClosestPoints(obj_id, ground_id, distance=0.002)
    if len(c2) > 0:
        contact_info[7:10] = c2[0][5]
        contact_info[10:13] = c2[0][7]
        contact_info[13]   = -c1[0][8] 

    # 推理下一帧
    xt_pred = infer_pose(xt, at, contact_info, model_v, model_w, dt, device='cpu')

    obj_pos = xt_pred[0:3]
    obj_rot = xt_pred[3:6]
    obj_v = xt_pred[6:9]
    obj_w = xt_pred[9:12]
    p.resetBasePositionAndOrientation(
        obj_id, obj_pos, R.from_rotvec(obj_rot).as_quat()
    )
    p.resetBasePositionAndOrientation(
        tcp_id, tcp_pos[k + 1], tcp_quat[k + 1]
    )
    p.performCollisionDetection()
    contact_info_ = np.zeros(7, dtype=np.float32)
    c1 = p.getClosestPoints(obj_id, tcp_id, distance=0.02)
    if len(c1) > 0:
        contact_info_[0:3] = c1[0][5]  # 接触点
        contact_info_[3:6] = c1[0][7]  # 法向量
        contact_info_[6]   = -c1[0][8]  # 接触深度

      # 输出
    print(f"Step {k}, Obj Pos: {xt_pred[0:3]}, Rotvec: {xt_pred[3:6]}, "
          f"Pred V: {xt_pred[6:9]}, Pred W: {xt_pred[9:12]}, Contact:{contact_info_[6]}")

    xt_err = xt_pred - xt_gt[k+1]
    print(f"Pos err:{xt_err[0:3]}, Rot err:{xt_err[3:6]}, "    
          f"V err:{xt_err[6:9]}, W err:{xt_err[9:12]}")
    
    # 修正
    xt_pred_ = correct_pred(xt_pred, contact_info_)

     # 输出
    obj_pos_ = xt_pred_[0:3]
    obj_rot_ = xt_pred_[3:6]  
    obj_v_ = xt_pred_[6:9]
    obj_w_ = xt_pred_[9:12]
    p.resetBasePositionAndOrientation(
        obj_id, obj_pos_, R.from_rotvec(obj_rot_).as_quat()
    )
    p.resetBasePositionAndOrientation(
        tcp_id, tcp_pos[k + 1], tcp_quat[k + 1]
    )
    p.performCollisionDetection()
    contact_info_1 = np.zeros(7, dtype=np.float32)
    c1 = p.getClosestPoints(obj_id, tcp_id, distance=0.02)
    if len(c1) > 0:
        contact_info_1[0:3] = c1[0][5]  # 接触点
        contact_info_1[3:6] = c1[0][7]  # 法向量
        contact_info_1[6]   = -c1[0][8]  # 接触深度
    print(f"Step {k}, Obj Pos: {xt_pred_[0:3]}, Rotvec: {xt_pred_[3:6]}, "
          f"Pred V: {xt_pred_[6:9]}, Pred W: {xt_pred_[9:12]},  Contact:{contact_info_1[6]}")

    xt_err = xt_pred_ - xt_gt[k+1]
    print(f"Pos err:{xt_err[0:3]}, Rot err:{xt_err[3:6]}, "
          f"V err:{xt_err[6:9]}, W err:{xt_err[9:12]}")

print("✅ Rollout 完成")

Step 0, Obj Pos: [ 0.04150889 -0.53613925  0.00771029], Rotvec: [-1.1850635  1.2503382  1.1400512], Pred V: [-1.4409118e-03 -3.6714427e-04 -5.2517457e-06], Pred W: [-9.1975118e-05  2.3450478e-04  8.8410489e-03], Contact:-0.00022293583606369793
Pos err:[5.30236935e-06 5.47476265e-05 1.37708156e-05], Rot err:[-0.00036251  0.00034982 -0.00010963], V err:[0.00012725 0.00131361 0.0003305 ], W err:[-0.01098848  0.00206615  0.00192995]
Step 0, Obj Pos: [ 0.04150889 -0.53613925  0.00771029], Rotvec: [-1.1850635  1.2503382  1.1400512], Pred V: [-1.4409118e-03 -3.6714427e-04 -5.2517457e-06], Pred W: [-9.1975118e-05  2.3450478e-04  8.8410489e-03],  Contact:-0.00022293583606369793
Pos err:[5.30236935e-06 5.47476265e-05 1.37708156e-05], Rot err:[-0.00036251  0.00034982 -0.00010963], V err:[0.00012725 0.00131361 0.0003305 ], W err:[-0.01098848  0.00206615  0.00192995]
Step 1, Obj Pos: [ 0.04138577 -0.536267    0.00769719], Rotvec: [-1.1851935  1.2496985  1.1406561], Pred V: [-2.8278360e-03 -1.752202